In [22]:
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import random

In [23]:
reproduciblity = True
if reproduciblity:
    random.seed(10)

In [24]:
import numpy as np
import pandas as pd


def feature_extractor_m3(data, time_resolution):
    """
    Feature extraction for M3 competition datasets.

    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing at least column ['y'].
        The index can be timestamp/year/month/etc.
    time_resolution : str
        One of: "Monthly", "Quarterly", "Yearly"

    Returns
    -------
    pd.DataFrame
        DataFrame with lag and rolling features.
    """

    data = data.copy()

    # Basic autoregressive lags
    data.loc[:, "y-1"] = data.loc[:, "y"].shift(1)
    data.loc[:, "y-2"] = data.loc[:, "y"].shift(2)
    data.loc[:, "y-3"] = data.loc[:, "y"].shift(3)

    if time_resolution == "monthly":

        # Seasonal lags for monthly data
        data.loc[:, "y-6"] = data.loc[:, "y"].shift(6)
        data.loc[:, "y-12"] = data.loc[:, "y"].shift(12)
        data.loc[:, "y-24"] = data.loc[:, "y"].shift(24)

        # Rolling features from y-1
        data.loc[:, "y-1_rolling_mean_3"] = data.loc[:, "y-1"].rolling(3).mean()
        data.loc[:, "y-1_rolling_std_3"] = data.loc[:, "y-1"].rolling(3).std()

        data.loc[:, "y-1_rolling_mean_6"] = data.loc[:, "y-1"].rolling(6).mean()
        data.loc[:, "y-1_rolling_std_6"] = data.loc[:, "y-1"].rolling(6).std()

        data.loc[:, "y-1_rolling_mean_12"] = data.loc[:, "y-1"].rolling(12).mean()
        data.loc[:, "y-1_rolling_std_12"] = data.loc[:, "y-1"].rolling(12).std()

        # Rolling features from seasonal lag y-12
        data.loc[:, "y-12_rolling_mean_3"] = data.loc[:, "y-12"].rolling(3).mean()
        data.loc[:, "y-12_rolling_std_3"] = data.loc[:, "y-12"].rolling(3).std()

        data.loc[:, "y-12_rolling_mean_6"] = data.loc[:, "y-12"].rolling(6).mean()
        data.loc[:, "y-12_rolling_std_6"] = data.loc[:, "y-12"].rolling(6).std()

        data.loc[:, "y-12_rolling_mean_12"] = data.loc[:, "y-12"].rolling(12).mean()
        data.loc[:, "y-12_rolling_std_12"] = data.loc[:, "y-12"].rolling(12).std()

    elif time_resolution == "quarterly":

        # Seasonal lags for quarterly data
        data.loc[:, "y-4"] = data.loc[:, "y"].shift(4)
        data.loc[:, "y-8"] = data.loc[:, "y"].shift(8)

        # Rolling features from y-1
        data.loc[:, "y-1_rolling_mean_2"] = data.loc[:, "y-1"].rolling(2).mean()
        data.loc[:, "y-1_rolling_std_2"] = data.loc[:, "y-1"].rolling(2).std()

        data.loc[:, "y-1_rolling_mean_4"] = data.loc[:, "y-1"].rolling(4).mean()
        data.loc[:, "y-1_rolling_std_4"] = data.loc[:, "y-1"].rolling(4).std()

        data.loc[:, "y-1_rolling_mean_8"] = data.loc[:, "y-1"].rolling(8).mean()
        data.loc[:, "y-1_rolling_std_8"] = data.loc[:, "y-1"].rolling(8).std()

        # Rolling features from seasonal lag y-4
        data.loc[:, "y-4_rolling_mean_2"] = data.loc[:, "y-4"].rolling(2).mean()
        data.loc[:, "y-4_rolling_std_2"] = data.loc[:, "y-4"].rolling(2).std()

        data.loc[:, "y-4_rolling_mean_4"] = data.loc[:, "y-4"].rolling(4).mean()
        data.loc[:, "y-4_rolling_std_4"] = data.loc[:, "y-4"].rolling(4).std()

    elif time_resolution == "yearly":

        # Yearly data has no fixed intra-year seasonality
        data.loc[:, "y-4"] = data.loc[:, "y"].shift(4)
        data.loc[:, "y-5"] = data.loc[:, "y"].shift(5)

        # Rolling features from recent yearly values
        data.loc[:, "y-1_rolling_mean_2"] = data.loc[:, "y-1"].rolling(2).mean()
        data.loc[:, "y-1_rolling_std_2"] = data.loc[:, "y-1"].rolling(2).std()

        data.loc[:, "y-1_rolling_mean_3"] = data.loc[:, "y-1"].rolling(3).mean()
        data.loc[:, "y-1_rolling_std_3"] = data.loc[:, "y-1"].rolling(3).std()

        data.loc[:, "y-1_rolling_mean_5"] = data.loc[:, "y-1"].rolling(5).mean()
        data.loc[:, "y-1_rolling_std_5"] = data.loc[:, "y-1"].rolling(5).std()

        # Rolling features from y-2
        data.loc[:, "y-2_rolling_mean_2"] = data.loc[:, "y-2"].rolling(2).mean()
        data.loc[:, "y-2_rolling_std_2"] = data.loc[:, "y-2"].rolling(2).std()

        data.loc[:, "y-2_rolling_mean_3"] = data.loc[:, "y-2"].rolling(3).mean()
        data.loc[:, "y-2_rolling_std_3"] = data.loc[:, "y-2"].rolling(3).std()

    else:
        raise ValueError(
            "time_resolution must be one of: 'Monthly', 'Quarterly', 'Yearly'"
        )

    # Remove rows with missing values caused by shift and rolling operations
    data = data.dropna()

    # Reset index like your M3 code
    data.index = np.arange(len(data))

    return data

In [25]:
m3s = {"monthly": 18, "quarterly": 8, "yearly": 6}

M3_TYPE = list(m3s.keys())[2]  # Change this to "quarterly" or "yearly" as needed
TEST_SIZE = m3s[M3_TYPE]
df = pd.read_csv(f"data/M3/M3_{M3_TYPE}_TSTS.csv")
os.mkdir(f"data/M3/Extracted/{M3_TYPE}/")

M3_TYPE
TEST_SIZE

6

In [26]:
# Make sure data is ordered correctly inside each time series
df = df.sort_values(["series_id", "timestamp"])
# Get unique series ids
unique_ids = df["series_id"].unique()
len(unique_ids)

645

In [27]:
n_series = min(1000, len(unique_ids))

random_ids = pd.Series(unique_ids).sample(
    n=n_series,
    random_state=42
).tolist()
len(random_ids)

645

In [28]:
# Filter dataframe
sample_df = df[df["series_id"].isin(random_ids)]
sample_df.head()

,series_id,category,value,timestamp
0,Y1,MICRO,940.66,1975
1,Y1,MICRO,1084.86,1976
2,Y1,MICRO,1244.98,1977
3,Y1,MICRO,1445.02,1978
4,Y1,MICRO,1683.17,1979


In [29]:
length_dict = {}
for series_id, group in sample_df.groupby("series_id"):
    print(f"Series ID: {series_id}, Length: {len(group)}")
    if len(group) in length_dict:
        length_dict[len(group)] += 1
    else:
        length_dict[len(group)] = 1
length_dict

Series ID: Y1, Length: 20
Series ID: Y10, Length: 20
Series ID: Y100, Length: 20
Series ID: Y101, Length: 20
Series ID: Y102, Length: 20
Series ID: Y103, Length: 20
Series ID: Y104, Length: 20
Series ID: Y105, Length: 20
Series ID: Y106, Length: 20
Series ID: Y107, Length: 20
Series ID: Y108, Length: 20
Series ID: Y109, Length: 20
Series ID: Y11, Length: 20
Series ID: Y110, Length: 20
Series ID: Y111, Length: 20
Series ID: Y112, Length: 20
Series ID: Y113, Length: 20
Series ID: Y114, Length: 20
Series ID: Y115, Length: 20
Series ID: Y116, Length: 20
Series ID: Y117, Length: 20
Series ID: Y118, Length: 20
Series ID: Y119, Length: 20
Series ID: Y12, Length: 20
Series ID: Y120, Length: 20
Series ID: Y121, Length: 20
Series ID: Y122, Length: 20
Series ID: Y123, Length: 20
Series ID: Y124, Length: 20
Series ID: Y125, Length: 20
Series ID: Y126, Length: 20
Series ID: Y127, Length: 20
Series ID: Y128, Length: 20
Series ID: Y129, Length: 20
Series ID: Y13, Length: 20
Series ID: Y130, Length: 2

{20: 152,
 32: 13,
 31: 3,
 44: 15,
 43: 10,
 47: 80,
 46: 20,
 21: 40,
 41: 9,
 34: 1,
 45: 6,
 33: 4,
 23: 103,
 36: 12,
 38: 2,
 22: 18,
 24: 5,
 35: 3,
 28: 3,
 42: 1,
 27: 2,
 30: 2,
 25: 129,
 26: 5,
 40: 1,
 37: 6}

In [30]:
sample_series_ids = list(sample_df["series_id"].unique())
len(sample_series_ids)

645

In [31]:
for series_id in sample_series_ids:
    series_data = sample_df[sample_df["series_id"] == series_id]
    series_data = series_data[["timestamp", "value"]]
    series_data = series_data.set_index("timestamp")
    series_data = series_data.rename(columns={"value": "y"})
    series_data = feature_extractor_m3(series_data, time_resolution=M3_TYPE)
    series_data.to_csv(f"data/M3/Extracted/{M3_TYPE}/{series_id}.csv", index=False)

